# Analyse

## Variables

- Quantitative
    - Continue
        - Température 
- Qualitative
    - Ordinal
        - Heure
    - Nominal
        - Class (Noisette ou Stitch)
        - Zone (Clapier, Cachette, Fontaine, ...)
- Autres
    - Date
        - Date du jour
        - Date de la capture
    - Coordonnées
        - Coordonnée du chon (en valeur normal [0:1] )

## Récupération des détections

In [2]:
from django.db import connection
from detections.models import Detection
from configuration.models import Zone, Family
from datetime import datetime
from plotly.graph_objects import FigureWidget
from IPython.display import display
from ipywidgets import HBox, VBox, Box, fixed, interactive_output

import os
import csv
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
import plotly.graph_objects as go

os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

In [3]:
detections_query =  'SELECT d.id, ' + \
                    'STRFTIME("%Y-%m-%d", c.date) AS date, ' + \
                    'STRFTIME("%Y-%m-%d %H:%M:%S", c.date) AS datetime, ' + \
                    'STRFTIME("%H", c.date) AS hour, ' + \
                    'STRFTIME("%Y%m%d%H", c.date) AS datehour_key, ' +  \
                       'z.id AS zone_id, ' +  \
                       'z.name AS zone_name, ' +  \
                       'f.[index] AS class_index, ' +  \
                       'f.name AS class_name, ' +  \
                       'd.center_x AS center_x_norm, ' +  \
                       'd.center_y AS center_y_norm ' +  \
                  'FROM detections_detection d ' +  \
                       'LEFT JOIN ' +  \
                       'detections_capture c ON d.capture_id = c.id ' +  \
                       'LEFT JOIN ' +  \
                       'configuration_family f ON d.family_id = f.id ' +  \
                       'LEFT JOIN ' +  \
                       'configuration_zone z ON d.zone_id = z.id ' +  \
                 'WHERE c.status == "archived" AND  ' +  \
                       'c.source == "vision" AND  ' +  \
                       '(f.[index] == 1 OR  ' +  \
                        'f.[index] == 2)  ' +  \
                 'ORDER BY c.date ASC '
                        
detections = Detection.objects.raw(detections_query)

zones_query = 'SELECT z.id, z.id AS zone_id, z.name AS zone_name FROM configuration_zone z ORDER BY z.id ASC'
zones = Zone.objects.raw(zones_query)

classes_query = 'SELECT f.id, f.[index] AS class_index, f.name AS class_name FROM configuration_family f ORDER BY f.id ASC'
classes = Family.objects.raw(classes_query)

def save_rows(rows, query, path): 
    columns = list()
    
    with connection.cursor() as cursor:
        cursor.execute(query)
        columns = [col[0] for col in cursor.description]
    
    with open(path, 'w+', newline='') as file:
        writer = csv.writer(file)

        writer.writerow(columns)
        
        for obj in rows:
            row = list()
            
            for col in columns:
                row.append(obj.__dict__[col])
                
            writer.writerow(row)
    
        print(f"{len(rows)} lignes exportées")

save_rows(detections, detections_query, 'Followchon/data/detections.csv')
save_rows(zones, zones_query, 'Followchon/data/zones.csv')
save_rows(classes, classes_query, 'Followchon/data/classes.csv')

53474 lignes exportées
14 lignes exportées
5 lignes exportées


In [4]:
df_detections = pd.read_csv('Followchon/data/detections.csv')
df_zones = pd.read_csv('Followchon/data/zones.csv')
df_classes = pd.read_csv('Followchon/data/classes.csv')

df_detections.sample(5)

,id,date,datetime,hour,datehour_key,zone_id,zone_name,class_index,class_name,center_x_norm,center_y_norm
4148,20903,2024-08-13,2024-08-13 20:46:11,20,2024081320,6.0,Bas,1,Noisette,0.632602,0.862150
28761,80290,2024-09-26,2024-09-26 09:45:01,9,2024092609,2.0,Cachette,2,Stitch,0.643555,0.212674
1934,13932,2024-08-07,2024-08-07 08:06:54,8,2024080708,6.0,Bas,1,Noisette,0.592448,0.571373
3867,19967,2024-08-11,2024-08-11 09:18:32,9,2024081109,12.0,Tunnel,2,Stitch,0.335749,0.453773
13108,46755,2024-09-08,2024-09-08 09:41:00,9,2024090809,12.0,Tunnel,1,Noisette,0.444824,0.266493


## Jointure avec les données météorologique

In [5]:
df_meteo = pd.read_csv('Followchon/data/H_69_latest-2023-2024.csv', sep=";")
df_meteo.sample(5)

,NUM_POSTE,NOM_USUEL,LAT,LON,ALTI,AAAAMMJJHH,RR1,QRR1,DRR1,QDRR1,...,INS,QINS,INS2,QINS2,TLAGON,QTLAGON,TVEGETAUX,QTVEGETAUX,ECOULEMENT,QECOULEMENT
65068,69114001,LIERGUES_SAPC,45.976500,4.650333,290,2023011917,NaN,1.0,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
39174,69029001,LYON-BRON,45.721333,4.949167,202,2023101322,0.0,1.0,0.0,9.0,...,0.0,9.0,0.0,9.0,NaN,NaN,NaN,NaN,NaN,NaN
4097,69008001,ANCY_SAPC,45.843333,4.508167,626,2023062017,0.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
57650,69066003,COURS LA VILLE_SAPC,46.098667,4.325833,578,2024011817,NaN,1.0,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
261106,69264001,VILLEFRANCHE,45.987167,4.737667,174,2023042001,0.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
detections_columns = ["datetime", "date", "hour", "zone_name", "zone_id", "class_name", "class_index"]
meteo_poste_columns = ['NUM_POSTE', 'NOM_USUEL', 'LAT', 'LON', 'ALTI']
meteo_temp_columns = ['AAAAMMJJHH', 'T']

mask_date = df_meteo["AAAAMMJJHH"] > df_detections['datehour_key'][0]
mask_lat_lon = df_meteo['LAT'].between(45.7, 45.8) & df_meteo['LON'].between(4.7, 5.1)

df_meteo_sorted = df_meteo.loc[mask_date & mask_lat_lon, meteo_poste_columns + meteo_temp_columns].sort_values(by=['AAAAMMJJHH', 'ALTI'])
df_meteo_reduced = df_meteo_sorted[meteo_temp_columns + meteo_poste_columns[-1:]].drop_duplicates(['AAAAMMJJHH'])

df_detections_t = pd.merge(df_detections, df_meteo_reduced, left_on="datehour_key", right_on="AAAAMMJJHH", how="left")[detections_columns + meteo_temp_columns[1:]]
df_detections_t = df_detections_t.rename(columns={'zone_name': 'zone', 'class_name' : 'class'})

df_detections_t.to_csv('Followchon/data/df_detections_t.csv', index=False)

df_detections_t.sample(5)

,datetime,date,hour,zone,zone_id,class,class_index,T
29274,2024-09-26 18:27:00,2024-09-26,18,Tunnel,12.0,Noisette,1,17.1
47068,2024-11-02 17:18:00,2024-11-02,17,NaN,NaN,Noisette,1,10.0
8605,2024-08-26 21:34:24,2024-08-26,21,Cachette,2.0,Stitch,2,20.5
37758,2024-10-10 09:19:00,2024-10-10,9,Clapier,1.0,Stitch,2,14.3
21016,2024-09-15 14:06:01,2024-09-15,14,Clapier,1.0,Noisette,1,18.7


In [7]:
df_detections_t = pd.read_csv('Followchon/data/df_detections_t.csv')

df_detections_t['date'] = pd.to_datetime(df_detections_t['datetime'])
df_detections_t['datetime'] = pd.to_datetime(df_detections_t['datetime'])

df_detections_t['zone'] = (df_detections_t['zone'].astype(str)).str.replace('nan', '')
df_detections_t['class'] = df_detections_t['class'].astype(str)

df_detections_t['T'] = pd.to_numeric(df_detections_t['T'])
df_detections_t['zone_id'] = pd.to_numeric(df_detections_t['zone_id'])
df_detections_t['class_index'] = pd.to_numeric(df_detections_t['class_index'])
df_detections_t['hour'] = pd.to_numeric(df_detections_t['hour'])

df_detections_t.sample(5)

,datetime,date,hour,zone,zone_id,class,class_index,T
29971,2024-09-27 18:14:01,2024-09-27 18:14:01,18,Clapier,1.0,Noisette,1,16.4
17975,2024-09-12 08:29:00,2024-09-12 08:29:00,8,Cachette,2.0,Noisette,1,12.7
16536,2024-09-11 08:31:18,2024-09-11 08:31:18,8,Fontaine,5.0,Noisette,1,15.4
21654,2024-09-15 19:24:00,2024-09-15 19:24:00,19,Tunnel maison,17.0,Noisette,1,15.5
37041,2024-10-08 13:39:01,2024-10-08 13:39:01,13,Cachette,2.0,Stitch,2,17.0


In [8]:
df_temp_by_date = df_detections_t.loc[(~df_detections_t['T'].isnull()), ['datetime', 'T']]\
    .groupby('datetime').mean()

df_mean_temp_by_date = df_temp_by_date.resample('D').mean().reset_index().rename(columns={'datetime' : 'date'})
df_mean_temp_by_date = df_mean_temp_by_date[~df_mean_temp_by_date['T'].isnull()]

df_mean_temp_by_date.to_csv('Followchon/data/df_mean_temp_by_date.csv', index=False)

df_temp_by_datehour = df_detections_t.loc[(~df_detections_t['T'].isnull()), ['datetime', 'T']]\
    .groupby('datetime').mean()

df_mean_temp_by_datehour = df_temp_by_datehour.resample('h').mean().reset_index().rename(columns={'datetime' : 'datehour'})
df_mean_temp_by_datehour = df_mean_temp_by_datehour[~df_mean_temp_by_datehour['T'].isnull()]

df_mean_temp_by_datehour.to_csv('Followchon/data/df_mean_temp_by_datehour.csv', index=False)

df_mean_temp_by_datehour.sample(5)

,datehour,T
453,2024-08-09 11:00:00,29.1
1589,2024-09-25 19:00:00,17.5
2397,2024-10-29 11:00:00,14.2
579,2024-08-14 17:00:00,26.2
606,2024-08-15 20:00:00,25.3


## Traitements des données

In [9]:
zones_all = ['Clapier', 'Cachette', 'Fontaine', 'Passerelle', 'Loft', 'Foin']
classes_all = ['Noisette', 'Stitch']
dates_all = df_detections_t['date'].unique()

hour_begin = 9
hour_end = 19
hours_all = list(range(hour_begin, hour_end + 1))

In [10]:
def calc_duration_by_zone(df, class_name):
    df_filterd = df.loc[df["class"] == class_name]
    
    set_duration_by_zone_and_day = dict()
    day_previous = None
    zone_previous = None
    datetime_zone_enter = None

    for i in df_filterd.index:
        day_current = df_filterd['datetime'][i].strftime('%Y-%m-%d')
        datetime_current = df_filterd['datetime'][i]
        zone_current = df_filterd['zone'][i]

        if day_previous is None or day_previous != day_current:
            day_previous = day_current
            zone_previous = None
            datetime_zone_enter = None
            set_duration_by_zone_and_day[day_previous] = {}

        if zone_previous is not None and datetime_zone_enter is not None and zone_previous != zone_current:
            zone_duration = datetime_current - datetime_zone_enter

            set_duration_by_zone_and_day[day_previous][zone_previous] = \
                zone_duration if zone_previous not in set_duration_by_zone_and_day[day_previous] \
                else set_duration_by_zone_and_day[day_previous][zone_previous] + zone_duration

        if zone_previous is None or zone_previous != zone_current:
            zone_previous = zone_current
            datetime_zone_enter = datetime_current

    df_duration_by_zone = pd.concat(
        { k: pd.DataFrame.from_dict(v, orient='index') for k, v in set_duration_by_zone_and_day.items() },
        axis=0,
    ).reset_index().rename(columns={'level_0': 'date', 'level_1': 'zone', 0: 'duration'})

    df_duration_by_zone['class'] = class_name
    df_duration_by_zone['date'] = pd.to_datetime(df_duration_by_zone['date'])
    df_duration_by_zone['duration'] = pd.to_timedelta(df_duration_by_zone['duration'], unit='s').dt.total_seconds() / 60
    
    df_duration_by_zone_merged = pd.merge(df_duration_by_zone, df_mean_temp_by_date, left_on="date", right_on="date", how="left")
    
    df_duration_by_zone_merged = pd.merge(df_duration_by_zone_merged, df_zones, left_on="zone", right_on="zone_name", how="left")
    df_duration_by_zone_merged = df_duration_by_zone_merged.drop(columns=['id', 'zone_name'])
    
    df_duration_by_zone_merged = pd.merge(df_duration_by_zone_merged, df_classes, left_on="class", right_on="class_name", how="left")
    df_duration_by_zone_merged = df_duration_by_zone_merged.drop(columns=['id', 'class_name'])
    
    df_duration_by_zone_filtered = df_duration_by_zone_merged[
        ~df_duration_by_zone_merged['zone'].eq('') 
        | ~df_duration_by_zone_merged['T'].isnull()
    ]

    return df_duration_by_zone_filtered

In [11]:
df_n_duration = calc_duration_by_zone(df_detections_t, classes_all[0])
df_s_duration = calc_duration_by_zone(df_detections_t, classes_all[1])

df_duration = pd.concat([df_n_duration ,df_s_duration], ignore_index=True)
                              
df_duration.to_csv('./data/df_duration.csv', index=False)

df_duration

,date,zone,duration,class,T,zone_id,class_index
0,2024-07-21,Clapier,18.300000,Noisette,25.210658,1.0,1
1,2024-07-21,,284.733333,Noisette,25.210658,NaN,1
2,2024-07-21,Cachette,141.700000,Noisette,25.210658,2.0,1
3,2024-07-21,Bas,8.466667,Noisette,25.210658,6.0,1
4,2024-07-22,,64.533333,Noisette,25.212821,NaN,1
...,...,...,...,...,...,...,...
1686,2024-11-18,Loft,1.016667,Stitch,NaN,14.0,2
1687,2024-11-19,Clapier,0.000000,Stitch,NaN,1.0,2
1688,2024-11-19,Cachette,13.000000,Stitch,NaN,2.0,2
1689,2024-11-19,Foin,10.000000,Stitch,NaN,16.0,2


In [12]:
df_duration_by_zone = df_duration.loc[:, ['zone', 'zone_id', 'class', 'class_index', 'duration']]\
    .groupby(['zone', 'zone_id', 'class', 'class_index'])\
    .mean('duration')\
    .reset_index()

df_duration_by_zone

,zone,zone_id,class,class_index,duration
0,Bas,6.0,Noisette,1,21.319792
1,Bas,6.0,Stitch,2,15.716026
2,Cachette,2.0,Noisette,1,223.564851
3,Cachette,2.0,Stitch,2,210.177723
4,Centre bas,10.0,Noisette,1,19.219271
5,Centre bas,10.0,Stitch,2,19.393889
6,Centre haut,9.0,Noisette,1,19.088384
7,Centre haut,9.0,Stitch,2,22.356061
8,Clapier,1.0,Noisette,1,153.072727
9,Clapier,1.0,Stitch,2,155.451852


In [155]:
def calc_occupation_by_zone(df, class_name):
    df_filterd = df.loc[df["class"] == class_name]
    
    set_duration_by_zone_and_hour_for_days = dict()
    day_previous = None
    zone_previous = None
    datetime_zone_enter = None

    for i in df.index:
        day_current = df['datetime'][i].strftime('%Y-%m-%d')
        datetime_current = df['datetime'][i]
        zone_current = df['zone'][i]

        if day_previous is None or day_previous != day_current:
            day_previous = day_current
            zone_previous = None
            datetime_zone_enter = None

        if zone_previous is not None and datetime_zone_enter is not None and zone_previous != zone_current:
            hour_enter = datetime_zone_enter.floor('h')
            hour_exit = datetime_current.floor('h')

            if day_previous not in set_duration_by_zone_and_hour_for_days:
                set_duration_by_zone_and_hour_for_days[day_previous] = {}

            if zone_previous not in set_duration_by_zone_and_hour_for_days[day_previous]:
                set_duration_by_zone_and_hour_for_days[day_previous][zone_previous] = {}

            hour_current = hour_enter
            while hour_current <= hour_exit:
                interval_begin = max(datetime_zone_enter, hour_current)
                interval_end = min(datetime_current, hour_current + pd.Timedelta(hours=1))

                interval = (interval_end - interval_begin).total_seconds() / 3600
                set_duration_by_zone_and_hour_for_days[day_previous][zone_previous][hour_current.hour] = \
                    interval if hour_current.hour not in set_duration_by_zone_and_hour_for_days[day_previous][zone_previous] \
                    else set_duration_by_zone_and_hour_for_days[day_previous][zone_previous][hour_current.hour] + interval

                hour_current += pd.Timedelta(hours=1)

        if zone_previous is None or zone_previous != zone_current:
            zone_previous = zone_current
            datetime_zone_enter = datetime_current

    df_occupation_by_zone = pd.DataFrame([
        {'date': date, 'datehour' : f"{date} {hour}:00:00", 'zone': zone, 'hour': hour, 'occupation': occupation}
        for date, by_date in set_duration_by_zone_and_hour_for_days.items()
        for zone, hours in by_date.items()
        for hour, occupation in hours.items()
    ])

    df_occupation_by_zone['class'] = class_name
    df_occupation_by_zone['date'] = pd.to_datetime(df_occupation_by_zone['date'])
    df_occupation_by_zone['datehour'] = pd.to_datetime(df_occupation_by_zone['datehour'])
    df_occupation_by_zone['hour'] = pd.to_numeric(df_occupation_by_zone['hour'])
    
    df_occupation_by_zone_merged = pd.merge(df_occupation_by_zone, df_mean_temp_by_datehour, left_on="datehour", right_on="datehour", how="left")
    
    df_occupation_by_zone_merged = pd.merge(df_occupation_by_zone_merged, df_zones, left_on="zone", right_on="zone_name", how="left")
    df_occupation_by_zone_merged = df_occupation_by_zone_merged.drop(columns=['id', 'zone_name'])
    
    df_occupation_by_zone_merged = pd.merge(df_occupation_by_zone_merged, df_classes, left_on="class", right_on="class_name", how="left")
    df_occupation_by_zone_merged = df_occupation_by_zone_merged.drop(columns=['id', 'class_name'])
    
    df_occupation_by_zone_filtered = df_occupation_by_zone_merged[
        ~(df_occupation_by_zone_merged['zone'].eq('') 
            | df_occupation_by_zone_merged['T'].isnull()
            | ~df_occupation_by_zone_merged['hour'].between(hour_begin, hour_end)
         )
    ]

    return df_occupation_by_zone_filtered

In [157]:
df_n_occupation = calc_occupation_by_zone(df_detections_t, classes_all[0])
df_s_occupation = calc_occupation_by_zone(df_detections_t, classes_all[1])

df_occupation = pd.concat([df_n_occupation ,df_s_occupation], ignore_index=True)
                              
df_occupation.to_csv('./data/df_occupation.csv', index=False)

df_occupation

,date,datehour,zone,hour,occupation,class,T,zone_id,class_index
0,2024-07-21,2024-07-21 15:00:00,Clapier,15,0.018056,Noisette,26.6,1.0,1
1,2024-07-21,2024-07-21 16:00:00,Clapier,16,0.003333,Noisette,25.5,1.0,1
2,2024-07-21,2024-07-21 17:00:00,Clapier,17,0.001111,Noisette,25.1,1.0,1
3,2024-07-21,2024-07-21 18:00:00,Clapier,18,0.005278,Noisette,25.1,1.0,1
4,2024-07-21,2024-07-21 19:00:00,Clapier,19,0.000278,Noisette,24.7,1.0,1
...,...,...,...,...,...,...,...,...,...
7149,2024-11-03,2024-11-03 16:00:00,Foin,16,0.099722,Stitch,13.2,16.0,2
7150,2024-11-03,2024-11-03 14:00:00,Tunnel maison,14,0.000000,Stitch,14.3,17.0,2
7151,2024-11-03,2024-11-03 14:00:00,Tunnel,14,0.000000,Stitch,14.3,12.0,2
7152,2024-11-03,2024-11-03 15:00:00,Tunnel,15,0.000000,Stitch,14.4,12.0,2


In [158]:
df_occupation_by_hour = df_occupation.loc[:, ['hour', 'class', 'class_index', 'zone', 'zone_id', 'occupation']]\
    .groupby(['hour', 'class', 'class_index', 'zone', 'zone_id'])\
    .mean('occupation')\
    .reset_index()

df_occupation_by_hour

,hour,class,class_index,zone,zone_id,occupation
0,9,Noisette,1,Bas,6.0,0.052755
1,9,Noisette,1,Cachette,2.0,0.413009
2,9,Noisette,1,Centre bas,10.0,0.059074
3,9,Noisette,1,Centre haut,9.0,0.051753
4,9,Noisette,1,Clapier,1.0,0.177119
...,...,...,...,...,...,...
299,19,Stitch,2,Maison,18.0,0.011905
300,19,Stitch,2,Passerelle,13.0,0.036466
301,19,Stitch,2,Piscine,4.0,0.015227
302,19,Stitch,2,Tunnel,12.0,0.064739


In [16]:
df_occupation_by_zone = df_occupation.loc[:, ['zone', 'zone_id', 'class', 'class_index', 'occupation']]\
    .groupby(['zone', 'zone_id', 'class', 'class_index'])\
    .mean('occupation')\
    .reset_index()

df_occupation_by_zone

,zone,zone_id,class,class_index,occupation
0,Bas,6.0,Noisette,1,0.069830
1,Bas,6.0,Stitch,2,0.069830
2,Cachette,2.0,Noisette,1,0.418285
3,Cachette,2.0,Stitch,2,0.418285
4,Centre bas,10.0,Noisette,1,0.069221
5,Centre bas,10.0,Stitch,2,0.069221
6,Centre haut,9.0,Noisette,1,0.047355
7,Centre haut,9.0,Stitch,2,0.047355
8,Clapier,1.0,Noisette,1,0.295786
9,Clapier,1.0,Stitch,2,0.295786


## Fonctions d'affichage

In [173]:
box_layout = widgets.Layout(
    display='flex',
    flex_flow='row wrap',  # Définit l'orientation et l'autorise à passer à la ligne
    justify_content='space-around',  # Espacement autour des items
    align_items='center',  # Aligne les items au centre verticalement
    width='100%',  # Largeur du conteneur
)

def generate_widget_corr(df, title, columns, columns_to_remove, size=700):
    df_corr = df.loc[:, columns].corr()

    for c in columns:
        df_corr.loc[c, c] = 0
        
    for c1 in columns_to_remove:
        for c2 in columns_to_remove:
            df_corr.loc[c1, c2] = 0
    
    fig = px.imshow(
        df_corr,
        color_continuous_scale="rdbu",
        title=str(title),
        zmin=-1,
        zmax=1,
        text_auto=".2f",
    )

    fig.update_layout(width=size, height=size)
    fig.update_traces(textfont_size=16)

    return FigureWidget(fig)

def generate_widget_scatter(df, title, x, y, point=30, width=500, height=500, withTrendline=False):
    fig = px.scatter(
        df, 
        x=x, 
        y=y, 
        title=str(title),
        trendline='lowess' if withTrendline else None,
        trendline_options=dict(frac=0.9) if withTrendline else dict(),
    )
    
    fig.update_traces(marker=dict(size=point))
    fig.update_layout(width=width,height=height)

    return FigureWidget(fig)

def generate_widget_histo(df, title, x, y, width=500, height=500):
    fig = px.histogram(
        df, 
        x=x, 
        y=y,
        title=str(title),
    )
    
    fig.update_layout(width=width, height=height)

    return FigureWidget(fig)

def generate_widget_box(df, title, x, y, width=500, height=500):
    fig = px.box(
        df, 
        x=x, 
        y=y,
        title=str(title),
    )
    
    fig.update_layout(width=width, height=height)

    return FigureWidget(fig)

def generate_widget_corr_zone(df, zone, columns, columns_to_remove, size=700):
    df_filtered = df.loc[df['zone'] == zone, columns]

    return generate_widget_corr(df_filtered, zone, columns, columns_to_remove, size)

def generate_widget_scatter_col(df, col, value, x, y, point=30, width=500, height=700, withTrendline=False):
    df_filtered = df.loc[df[col] == value]

    return generate_widget_scatter(df_filtered, value, x, y, point, width, height, withTrendline)

def generate_widget_histo_col(df, col, value, x, y, width=500, height=700):
    df_filtered = df.loc[df[col] == value]

    return generate_widget_histo(df_filtered, value, x, y, width, height)

def generate_widget_box_col(df, col, value, x, y, width=500, height=700):
    df_filtered = df.loc[df[col] == value]

    return generate_widget_box(df_filtered, value, x, y, width, height)

def generate_widget_corr_class(df, class_name, columns, columns_to_remove, size=700):
    df_filtered = df.loc[df['class'] == class_name, columns]

    return generate_widget_corr(df_filtered, class_name, columns, columns_to_remove, size)

def generate_widget_corr_hour(df, hour, columns, columns_to_remove, size=700):
    df_filtered = df.loc[df['hour'] == hour, columns]

    return generate_widget_corr(df_filtered, hour, columns, columns_to_remove, size)

def generate_widget_corr_by_class(df, columns, columns_to_remove, size=700):
    widgets = list()
       
    for class_name in classes_all:
        widgets.append(
            generate_widget_corr_class(
                df, 
                class_name=class_name, 
                columns=list(filter(lambda c: c != 'class' and c != 'class_index', columns)),
                columns_to_remove=columns_to_remove,
                size=size
            )
        )

    return widgets

def generate_widget_corr_by_zone(df, columns, columns_to_remove, size=500):
    widgets = list()
       
    for zone in zones_all:
        widgets.append(
            generate_widget_corr_zone(
                df, 
                zone, 
                list(filter(lambda c: c != 'zone' and c != 'zone_id', columns)),
                columns_to_remove,
                size=size,
            )
        )

    return widgets

def generate_widget_scatter_by_col(df, col, x, y, point=30, width=500, height=500, withTrendline=False):
    widgets = list()
       
    for value in df[col].unique():
        widgets.append(
            generate_widget_scatter_col(
                df, 
                col,
                value, 
                x,
                y,
                point=point,
                width=width,
                height=height,
                withTrendline=withTrendline
            )
        )

    return widgets

def generate_widget_histo_by_col(df, col, x, y, width=500, height=500):
    widgets = list()
       
    for value in df[col].unique():
        widgets.append(
            generate_widget_histo_col(
                df, 
                col,
                value, 
                x,
                y,
                width=width,
                height=height,
            )
        )

    return widgets

def generate_widget_box_by_col(df, col, x, y, width=500, height=500):
    widgets = list()
       
    for value in df[col].unique():
        widgets.append(
            generate_widget_box_col(
                df, 
                col,
                value, 
                x,
                y,
                width=width,
                height=height,
            )
        )

    return widgets

def generate_widget_corr_by_hour(df, columns, columns_to_remove, size=500):
    widgets = list()
       
    for hour in hours_all:
        widgets.append(
            generate_widget_corr_by_hour(
                df, 
                hour, 
                list(filter(lambda c: c != 'hour', columns)),
                columns_to_remove,
                size=size,
            )
        )

    return widgets

def generate_box(children, layout):
    return Box(
        children=children, 
        layout=layout
    )

## Recherche de corrélation

### Par class

#### Sans regroupements

In [174]:
display(
    generate_box(
        children=generate_widget_corr_by_class(df_detections_t, columns=['datetime', 'hour', 'T', 'zone_id', 'class_index'], columns_to_remove=['datetime', 'T'], size=700),
        layout=box_layout
    ),
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

#### Avec durée par zone et jour

In [175]:
display(
    generate_box(
        children=generate_widget_corr_by_class(df_duration, ['date', 'duration', 'T', 'zone_id', 'class_index'], ['date', 'T'], 700),
        layout=box_layout
    )
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

#### Avec durée par zone

In [176]:
display(
    generate_box(
        children=generate_widget_corr_by_class(df_duration_by_zone, ['duration', 'zone_id', 'class_index'], [], 700),
        layout=box_layout
    )
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

#### Corrélation zone / durée

In [177]:
df_duration_filtered = df_duration[
    df_duration['zone'].str.contains('|'.join(zones_all))
].sort_values('zone_id')

display(
    generate_box(
        children=[
            generate_widget_histo(df_duration_filtered, 'Zone/Duration', 'zone', 'duration', 800, 700),
            generate_widget_box(df_duration_filtered, 'Zone/Duration', 'zone', 'duration', 800, 700),
        ],
        layout=box_layout
    )
)

Box(children=(FigureWidget({
    'data': [{'alignmentgroup': 'True',
              'bingroup': 'x',
          …

#### Avec occupation par zone et par heure

In [178]:
display(
    generate_box(
        children=generate_widget_corr_by_class(df_occupation, ['date', 'hour', 'occupation', 'T', 'zone_id', 'class_index'], ['date', 'hour', 'T'], 700),
        layout=box_layout
    )
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

#### Avec occupation par heure

In [179]:
display(
    generate_box(
        children=generate_widget_corr_by_class(df_occupation_by_hour, ['hour', 'occupation', 'zone_id', 'class_index'], [], 700),
        layout=box_layout
    )
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

#### Avec occupation par zone

In [180]:
display(
    generate_box(
        children=generate_widget_corr_by_class(df_occupation_by_zone, ['occupation', 'zone_id', 'class_index'], [], 700),
        layout=box_layout
    )
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

#### Corrélation Zone / Occupation

In [181]:
df_occupation_filtered = df_occupation[
    df_occupation['zone'].str.contains('|'.join(zones_all))
].sort_values('zone_id')

display(
    generate_box(
        children=[
            generate_widget_histo(df_occupation_filtered, 'Zone/Occupation', 'zone', 'occupation', 800, 700),
            generate_widget_box(df_occupation_filtered, 'Zone/Occupation', 'zone', 'occupation', 800, 700),
        ],
        layout=box_layout
    )
)

Box(children=(FigureWidget({
    'data': [{'alignmentgroup': 'True',
              'bingroup': 'x',
          …

### Par zone

#### Sans regroupements

In [97]:
display(
    generate_box(
        children=generate_widget_corr_by_zone(df_detections_t, ['datetime', 'hour', 'T', 'class_index', 'zone_id'], ['datetime', 'T', 'hour'], 450), 
        layout=box_layout
    ),
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

#### Avec durée par zone et jour

In [25]:
display(
    generate_box(
        children=generate_widget_corr_by_zone(df_duration, ['date', 'duration', 'T', 'class_index', 'zone_id'], ['date', 'T'], 450),
        layout=box_layout
    )
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

#### Corrélation date/duration par zone

In [147]:
df_duration_filtered = df_duration[
    df_duration['zone'].str.contains('|'.join(['Clapier', 'Cachette', 'Foin']))
].sort_values('zone_id')

display(
    generate_box(
        children=generate_widget_scatter_by_col(df_duration_filtered, 'zone', 'date', 'duration', 10, 500, 500, True), 
        layout=box_layout
    ),
    generate_box(
        children=generate_widget_histo_by_col(df_duration_filtered, 'zone', 'date', 'duration', 500, 500), 
        layout=box_layout
    )
)

Box(children=(FigureWidget({
    'data': [{'hovertemplate': 'date=%{x}<br>duration=%{y}<extra></extra>',
     …

Box(children=(FigureWidget({
    'data': [{'alignmentgroup': 'True',
              'bingroup': 'x',
          …

#### Corrélaton T/duration par zone

In [146]:
df_duration_filtered = df_duration[
    df_duration['zone'].str.contains('|'.join(['Clapier', 'Cachette', 'Foin']))
].sort_values('zone_id')

display(
    generate_box(
        children=generate_widget_scatter_by_col(df_duration_filtered, 'zone', 'T', 'duration', 10, 500, 500, True), 
        layout=box_layout
    ),
    generate_box(
        children=generate_widget_histo_by_col(df_duration_filtered, 'zone', 'T', 'duration', 500, 500), 
        layout=box_layout
    ),
)

Box(children=(FigureWidget({
    'data': [{'hovertemplate': 'T=%{x}<br>duration=%{y}<extra></extra>',
        …

Box(children=(FigureWidget({
    'data': [{'alignmentgroup': 'True',
              'bingroup': 'x',
          …

#### Avec durée par zone

In [26]:
display(
    generate_box(
        children=generate_widget_corr_by_zone(df_duration_by_zone, ['duration', 'class_index', 'zone_id'], [], 450),
        layout=box_layout
    )
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

#### Avec occupation par zone et par heure

In [182]:
display(
    generate_box(
        children=generate_widget_corr_by_zone(df_occupation, ['date', 'hour', 'occupation', 'T', 'class_index', 'zone_id'], ['date', 'hour', 'T'], 450),
        layout=box_layout
    )
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

#### Occupation par heure par zone

In [183]:
display(
    generate_box(
        children=generate_widget_corr_by_zone(df_occupation_by_hour, ['hour', 'occupation', 'zone_id', 'class_index'], [], 450),
        layout=box_layout
    )
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

#### Corrélation heure/occupation par zone

In [187]:
df_occupation_by_hour_filtered = df_occupation_by_hour[
    df_occupation_by_hour['zone'].str.contains('|'.join(['Clapier', 'Fontaine', 'Loft', 'Cachette', 'Foin']))
    & df_occupation_by_hour['hour'].between(hour_begin, hour_end)
].sort_values('zone_id')


display(
    generate_box(
        children=generate_widget_scatter_by_col(df_occupation_by_hour_filtered, 'zone', 'hour', 'occupation', 10, 340, 500, True), 
        layout=box_layout
    ),
    generate_box(
        children=generate_widget_histo_by_col(df_occupation_by_hour_filtered, 'zone', 'hour', 'occupation', 340, 500), 
        layout=box_layout
    ),
)

Box(children=(FigureWidget({
    'data': [{'hovertemplate': 'hour=%{x}<br>occupation=%{y}<extra></extra>',
   …

Box(children=(FigureWidget({
    'data': [{'alignmentgroup': 'True',
              'bingroup': 'x',
          …

### Par heure

#### Sans regroupement

In [29]:
display(
    generate_box(
        children=generate_widget_corr_by_hour(df_detections_t, ['datetime', 'T', 'class_index', 'zone_id', 'hour'], ['datetime', 'T'], 360), 
        layout=box_layout
    ),
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

#### Avec occupation par zone et par heure

In [30]:
display(
    generate_box(
        children=generate_widget_corr_by_hour(df_occupation, ['date', 'occupation', 'T', 'class_index', 'zone_id', 'hour'], ['date', 'T'], 360),
        layout=box_layout
    )
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…

#### Corrélation zone/occupation par heure

In [151]:
df_occupation_filtered = df_occupation[
        df_occupation['hour'].between(hour_begin, hour_end)
    ].sort_values('hour')

display(
    generate_box(
        children=generate_widget_scatter_by_col(df_occupation_filtered, 'hour', 'zone', 'occupation', 10, 360, 360), 
        layout=box_layout
    ),
    generate_box(
        children=generate_widget_histo_by_col(df_occupation_filtered, 'hour', 'zone', 'occupation', 360, 360), 
        layout=box_layout
    ),
)

Box(children=(FigureWidget({
    'data': [{'hovertemplate': 'zone=%{x}<br>occupation=%{y}<extra></extra>',
   …

Box(children=(FigureWidget({
    'data': [{'alignmentgroup': 'True',
              'bingroup': 'x',
          …

#### Avec occupation par heure

In [31]:
display(
    generate_box(
        children=generate_widget_corr_by_hour(df_occupation_by_hour, ['hour', 'occupation', 'zone_id', 'class_index'], [], 360),
        layout=box_layout
    )
)

Box(children=(FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<b…